# Phase 2 — Deep Learning Anomaly Detection
## Stage 4: Architecture Design & Theoretical Foundation

**Project:** Adaptive Cyber-Physical Security  
**Dataset:** CICIDS-2017 (2.83M flows, 34 features after Phase 1 preprocessing)  
**Phase 1 Baseline:** GMM K=12, F1=90.97%, AUC=95.76%  
**Phase 2 Goal:** Build Model B — sequence-based deep learning anomaly detector  

---

## Section 1: Theoretical Foundations

### 1.1 — Why Sequence Modeling for Intrusion Detection?

#### The IID Assumption Failure in Phase 1

Phase 1's GMM treated every network flow as an **independent and identically distributed (IID)** sample drawn from a stationary distribution. This is a convenient mathematical fiction, but network traffic is fundamentally *not* IID. Flows are causally ordered in time: a SYN packet precedes a SYN-ACK, a login attempt precedes a data exfiltration, a reconnaissance scan precedes an exploitation attempt. By discarding this ordering, Phase 1 discards the single richest source of attack signal.

The empirical evidence from Phase 1 confirms the failure: **Bot attacks were detected at only 45.2%**, and **Infiltration attacks at 33.1%**. Both are canonical low-and-slow attack classes whose defining characteristic is *temporal spread* — they deliberately fragment their malicious behaviour across dozens or hundreds of flows, each of which, viewed in isolation, appears entirely benign.

#### What a Low-and-Slow Attack Looks Like Across 50 Flows

Consider a **bot beaconing** scenario over a 50-flow window sampled at typical enterprise network rates (~10 flows per minute = approximately 5 minutes of traffic):

| Flow index | Individual behaviour | Isolated anomaly score |
|-----------|----------------------|------------------------|
| 1–5       | Normal HTTP GET to internal server | Low |
| 6–10      | Slightly elevated `Bwd IAT Std` — C2 polling | Low |
| 11–20     | Normal DNS queries — but to unusual TLD | Low |
| 21–35     | Periodic 300-second intervals in `Flow IAT Std` | Low |
| 36–50     | Small consistent payload size — encrypted heartbeat | Low |

No single flow in this window crosses a GMM anomaly threshold. But the *sequence* encodes a distinct pattern: **strict periodicity of inter-arrival times, consistent payload entropy, and directional asymmetry persisting across the entire window**. A sequence model trained exclusively on benign traffic will fail to reconstruct this pattern faithfully — its reconstruction error for this 50-flow window will be significantly elevated above the benign baseline.

#### Temporal Anomaly: Individually Normal, Collectively Anomalous

We define a **temporal anomaly** as a sequence $\mathbf{X}^{(t)} = [\mathbf{x}_t, \mathbf{x}_{t+1}, \ldots, \mathbf{x}_{t+W-1}]$ where each individual flow $\mathbf{x}_i$ lies within the benign marginal distribution, yet the joint distribution $p(\mathbf{x}_t, \mathbf{x}_{t+1}, \ldots, \mathbf{x}_{t+W-1})$ deviates significantly from the benign joint distribution.

Formally, let $\mathcal{M}_{\text{benign}}$ denote the manifold of benign sequences learned by a deep autoencoder. A sequence is anomalous if:

$$s(\mathbf{X}^{(t)}) = \frac{1}{W} \left\| \mathbf{X}^{(t)} - \hat{\mathbf{X}}^{(t)} \right\|^2 > \tau$$

where $\hat{\mathbf{X}}^{(t)} = f_{\text{AE}}(\mathbf{X}^{(t)})$ is the autoencoder's reconstruction and $\tau$ is a threshold calibrated on held-out benign sequences. The key property: **the autoencoder is trained only on benign sequences**, so it learns to reconstruct the benign joint distribution. Attack sequences, which inhabit a different region of sequence space, cannot be faithfully reconstructed — their residuals $\|\mathbf{x}_i - \hat{\mathbf{x}}_i\|^2$ accumulate across the window into a detectably large aggregate score.

#### Connection to Phase 1 Failure Analysis

Phase 1's failure on Bot attacks (45.2% detection) is precisely explained by this mechanism. Bot C2 communication operates at low rates over extended periods — each individual beacon flow has normal-looking feature values for `Flow Bytes/s`, `Fwd Packet Length Mean`, and `Bwd IAT Min`. The GMM's per-flow log-likelihood correctly assigns these flows to benign clusters. But the **temporal fingerprint** — the strict 300-second periodicity, the consistent payload ratio, the unchanging `Init_Win_bytes_backward` across 20+ consecutive flows — is completely invisible to a model that processes flows independently.

Similarly, Infiltration attacks (33.1% Phase 1 detection) involve multi-stage lateral movement: reconnaissance, exploitation, and data exfiltration unfold across hundreds of flows, each stage individually plausible but collectively forming a definitive attack narrative.

**Sequence modeling directly targets this gap.** By processing $W=50$ consecutive flows as a single unit, both the LSTM Autoencoder and Transformer Autoencoder proposed in Phase 2 can capture:
1. Short-range dependencies (inter-packet timing within a burst)
2. Medium-range periodicities (C2 beacon intervals within a 5-minute window)
3. Long-range structural anomalies (persistent directional asymmetry across the window)

### 1.2 — LSTM Autoencoder Theory

#### A) LSTM Cell Equations

The Long Short-Term Memory (LSTM) cell (Hochreiter & Schmidhuber, 1997) addresses the vanishing gradient problem of vanilla RNNs through a gating mechanism that explicitly controls information flow. For hidden state dimension $d_h$ and input dimension $d_x$, the cell equations at time step $t$ are:

**Forget gate** — decides what to discard from the previous cell state:
$$\mathbf{f}_t = \sigma\!\left(\mathbf{W}_f [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_f\right)$$

**Input gate** — decides which new information to store:
$$\mathbf{i}_t = \sigma\!\left(\mathbf{W}_i [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_i\right)$$

**Candidate cell update** — computes new candidate values:
$$\tilde{\mathbf{C}}_t = \tanh\!\left(\mathbf{W}_C [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_C\right)$$

**Cell state update** — element-wise gated combination:
$$\mathbf{C}_t = \mathbf{f}_t \odot \mathbf{C}_{t-1} + \mathbf{i}_t \odot \tilde{\mathbf{C}}_t$$

**Output gate** — decides what to expose as the hidden state:
$$\mathbf{o}_t = \sigma\!\left(\mathbf{W}_o [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_o\right)$$

**Hidden state** — the output passed to the next layer:
$$\mathbf{h}_t = \mathbf{o}_t \odot \tanh(\mathbf{C}_t)$$

where $\sigma$ is the sigmoid function, $\odot$ denotes element-wise (Hadamard) product, and $[\mathbf{h}_{t-1}, \mathbf{x}_t]$ denotes concatenation. The weight matrices $\mathbf{W}_f, \mathbf{W}_i, \mathbf{W}_C, \mathbf{W}_o \in \mathbb{R}^{d_h \times (d_h + d_x)}$ are learned during training.

#### B) Why LSTM Beats Simple RNN for This Task

A vanilla RNN computes $\mathbf{h}_t = \tanh(\mathbf{W}[\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b})$. During backpropagation through time (BPTT), gradients are multiplied by $\mathbf{W}$ at every time step. For sequences of length $W=50$, the gradient signal at step $t=1$ is proportional to $\mathbf{W}^{50}$: if the spectral radius of $\mathbf{W}$ is less than 1, gradients vanish exponentially; if greater than 1, they explode. In both cases, the network cannot learn dependencies spanning more than ~10 steps.

The LSTM's **forget gate** $\mathbf{f}_t$ solves this directly. The cell state gradient flows through:
$$\frac{\partial \mathbf{C}_t}{\partial \mathbf{C}_{t-1}} = \mathbf{f}_t$$

Since $\mathbf{f}_t \in (0, 1)^{d_h}$ is learned, the network can set $f_{t,k} \approx 1$ for dimensions $k$ where long-range dependencies matter — creating a **constant error carousel** that propagates gradients over arbitrarily long spans without decay.

For our intrusion detection task, this is critical: **LSTM can learn to forget irrelevant old flows and retain relevant patterns (e.g., a port scan pattern from 20 flows ago)**. When processing a 50-flow window containing a slow port scan (one probe per 3 flows), the LSTM encoder can maintain a running count of unique destination ports in its cell state, accumulating evidence across the entire window before compressing to the latent vector.

#### C) Autoencoder Principle

An autoencoder is a neural network trained to map its input to itself via a constrained bottleneck. For sequence inputs $\mathbf{X} \in \mathbb{R}^{W \times F}$:

**Encoder** compresses the sequence to a latent representation:
$$\mathbf{z} = f_{\text{enc}}(\mathbf{X}) \in \mathbb{R}^d, \quad d \ll W \cdot F$$

With $W=50$, $F=34$, and $d=32$: the bottleneck compresses $1700 \to 32$ dimensions (compression ratio $\approx 53\times$), forcing the network to learn a compact generative model of benign sequences.

**Decoder** reconstructs the sequence from the latent code:
$$\hat{\mathbf{X}} = f_{\text{dec}}(\mathbf{z}) \in \mathbb{R}^{W \times F}$$

**Training loss** (Mean Squared Error over all flows and features):
$$\mathcal{L}_{\text{train}} = \frac{1}{N} \sum_{n=1}^{N} \left\| \mathbf{X}^{(n)} - \hat{\mathbf{X}}^{(n)} \right\|_F^2$$

where $\|\cdot\|_F$ is the Frobenius norm and $N$ is the number of training sequences.

**Anomaly score** for a test sequence:
$$s(\mathbf{X}) = \frac{1}{W} \left\| \mathbf{X} - f_{\text{AE}}(\mathbf{X}) \right\|_F^2 = \frac{1}{W} \sum_{t=1}^{W} \left\| \mathbf{x}_t - \hat{\mathbf{x}}_t \right\|_2^2$$

**Detection threshold** calibrated on validation benign sequences:
$$\tau = \mu_{\text{benign}} + k \cdot \sigma_{\text{benign}}$$

where $\mu_{\text{benign}}$ and $\sigma_{\text{benign}}$ are the mean and standard deviation of anomaly scores on held-out benign sequences, and $k$ is a scalar hyperparameter tuned to trade off precision vs. recall (typically $k \in [2, 5]$).

#### D) Why Autoencoders for Anomaly Detection

The key insight is the **manifold hypothesis**: benign network traffic lies on (or near) a low-dimensional manifold $\mathcal{M}_{\text{benign}} \subset \mathbb{R}^{W \times F}$. An autoencoder trained exclusively on benign sequences learns to project inputs onto this manifold. For any benign test sequence $\mathbf{X}_{\text{benign}} \in \mathcal{M}_{\text{benign}}$, the projection is faithful: $\hat{\mathbf{X}} \approx \mathbf{X}$, so $s(\mathbf{X}_{\text{benign}}) \approx 0$.

Attack sequences $\mathbf{X}_{\text{attack}} \notin \mathcal{M}_{\text{benign}}$ cannot be faithfully projected onto the benign manifold. The decoder, trained only to reconstruct benign patterns, produces $\hat{\mathbf{X}} \approx \text{proj}_{\mathcal{M}_{\text{benign}}}(\mathbf{X}_{\text{attack}})$, which differs from the attack input by a non-trivial residual: $s(\mathbf{X}_{\text{attack}}) \gg 0$.

This is the **same principle as Phase 1's GMM** — both model the benign distribution and flag deviations — but the LSTM-AE operates in **sequence space** rather than per-flow space, making it sensitive to temporal anomalies that Phase 1 cannot detect.

### 1.3 — Transformer Autoencoder Theory

#### A) Self-Attention Mechanism

The Transformer (Vaswani et al., 2017) replaces sequential processing with a **global attention** mechanism that relates every position in the sequence to every other position in a single operation. Given an input sequence $\mathbf{X} \in \mathbb{R}^{W \times d_{\text{model}}}$, the attention mechanism computes three projections:

$$\mathbf{Q} = \mathbf{X}\mathbf{W}_Q, \quad \mathbf{K} = \mathbf{X}\mathbf{W}_K, \quad \mathbf{V} = \mathbf{X}\mathbf{W}_V$$

$$\text{Attention}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{softmax}\!\left(\frac{\mathbf{Q}\mathbf{K}^\top}{\sqrt{d_k}}\right) \mathbf{V}$$

where $\mathbf{W}_Q, \mathbf{W}_K, \mathbf{W}_V \in \mathbb{R}^{d_{\text{model}} \times d_k}$ are learned projection matrices. The scaling factor $\frac{1}{\sqrt{d_k}}$ prevents the dot products from growing large in magnitude (which would push softmax into saturation regions with vanishing gradients).

**Interpretation for network flows:**
- **Queries ($\mathbf{Q}$):** "What information does flow at position $t$ need from the context?" For flow $t$, the query asks: what features of past or future flows are relevant to understanding whether *this* flow is anomalous?
- **Keys ($\mathbf{K}$):** "What information does each flow offer to queries?" The key of flow $s$ encodes what it can contribute to other flows' context.
- **Values ($\mathbf{V}$):** "What is the actual content to retrieve?" The value is the feature representation that gets aggregated and passed forward.

The output at position $t$ is a weighted sum of all values, where weights are determined by the similarity of query $t$ to all keys. High attention weight between flow $t$ and flow $s$ means: "flow $t$ is highly influenced by the temporal context of flow $s$."

#### B) Multi-Head Attention

A single attention head can only attend to one type of relationship at a time. **Multi-head attention** runs $h$ attention heads in parallel, each with its own projections:

$$\text{head}_i = \text{Attention}(\mathbf{Q}\mathbf{W}_Q^{(i)},\ \mathbf{K}\mathbf{W}_K^{(i)},\ \mathbf{V}\mathbf{W}_V^{(i)})$$

$$\text{MultiHead}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h)\mathbf{W}_O$$

where $\mathbf{W}_O \in \mathbb{R}^{hd_v \times d_{\text{model}}}$ projects the concatenated heads back to the model dimension.

**Why multiple heads for intrusion detection?** Different heads can specialise in different temporal scales and relationship types. With $h=4$ heads:
- **Head 1** may specialise in short-range burst structure (flows $t$ and $t\pm1$)
- **Head 2** may attend to periodic patterns — e.g., flow $t$ attends strongly to flow $t-10$ if C2 beaconing repeats every 10 flows
- **Head 3** may capture directional asymmetry relationships (flows with high `Bwd Packets/s` attending to preceding flows with high `Fwd Packet Length Mean`)
- **Head 4** may encode global statistics — every flow attending to the sequence mean

This representational richness is impossible with a single attention head and is one of the primary advantages of Transformers over LSTMs for detecting multi-scale temporal anomalies.

#### C) Positional Encoding

Unlike LSTMs, the Transformer's attention mechanism is **permutation-equivariant**: if you shuffle the input flows, the output is shuffled identically (the attention weights change, but the network has no inherent notion of order). For flow sequences, **order is causally meaningful** — flow 1 preceded flow 2 in real time, and that ordering carries information (e.g., a SYN before a SYN-ACK, a probe before a response).

To inject positional information, we add sinusoidal positional encodings to the input:

$$\text{PE}(\text{pos}, 2i) = \sin\!\left(\frac{\text{pos}}{10000^{2i/d_{\text{model}}}}\right)$$

$$\text{PE}(\text{pos}, 2i+1) = \cos\!\left(\frac{\text{pos}}{10000^{2i/d_{\text{model}}}}\right)$$

where $\text{pos} \in \{0, 1, \ldots, W-1\}$ is the flow's position in the window and $i \in \{0, 1, \ldots, d_{\text{model}}/2 - 1\}$ indexes the encoding dimension. The sinusoidal form was chosen by Vaswani et al. because it allows the model to generalise to relative positions: $\text{PE}(\text{pos}+k)$ can be expressed as a linear function of $\text{PE}(\text{pos})$ for any fixed offset $k$, enabling the attention mechanism to learn relative position dependencies.

For our task with $W=50$: the positional encoding ensures that flow 1's contribution to the latent representation is distinct from flow 50's, even if their feature vectors are identical. This is critical for detecting attacks where the **ordering** of events (not just their co-occurrence) is the anomaly signal.

#### D) Flash Attention — Memory-Efficient Attention for Long Sequences

**Standard attention** computes and materialises the full $W \times W$ attention matrix in GPU memory:
$$\mathbf{S} = \frac{\mathbf{Q}\mathbf{K}^\top}{\sqrt{d_k}} \in \mathbb{R}^{W \times W}$$

Memory complexity: $O(W^2 \cdot d_k)$. For $W=50$, this is $50^2 \cdot 16 = 40{,}000$ floats — trivial. For longer sequences, however, the quadratic scaling becomes prohibitive: $W=1{,}000$ requires 4M floats per head per layer, and $W=10{,}000$ requires 400M floats (1.6 GB at float32 for a single-head single-layer).

**Flash Attention** (Dao et al., 2022) reformulates the attention computation to avoid materialising $\mathbf{S}$ in full. Using a **tiled computation** that streams blocks of $\mathbf{Q}$ and $\mathbf{K}$ from HBM (high-bandwidth memory) to SRAM (fast on-chip memory), it computes the attention output while maintaining only $O(W)$ working memory:
1. Partition $\mathbf{Q}$ into blocks of size $B_r$ and $\mathbf{K}, \mathbf{V}$ into blocks of size $B_c$
2. For each query block, compute partial attention scores with all key blocks incrementally
3. Maintain running softmax normalisation using the online softmax trick (Milakov & Gimelshein, 2018)
4. Accumulate output directly without ever writing the full $W \times W$ matrix

**For our Phase 2 implementation** ($W=50$): standard attention is perfectly adequate — the full attention matrix fits comfortably in memory. Flash Attention becomes relevant when extending this work to longer windows (e.g., $W=500$ for hour-long traffic captures) or larger batch sizes. The architectural awareness is, however, important: if Phase 3 hybrid experiments reveal that $W=50$ is insufficient for detecting very slow attacks (e.g., weekly exfiltration patterns), Flash Attention enables scaling to $W=5{,}000$ without memory constraints.

#### E) Transformer Autoencoder — Encoder-Decoder Wrapper

The full Transformer-AE wraps the attention mechanism in an encoder-decoder structure:

**Encoder:**
$$\mathbf{X}' = \mathbf{X} + \text{PE} \quad \text{(add positional encoding)}$$
$$\mathbf{H} = \text{LayerNorm}(\mathbf{X}' + \text{MultiHead}(\mathbf{X}', \mathbf{X}', \mathbf{X}')) \quad \text{(self-attention + residual)}$$
$$\mathbf{H} = \text{LayerNorm}(\mathbf{H} + \text{FFN}(\mathbf{H})) \quad \text{(feed-forward + residual)}$$
$$\mathbf{z} = \text{Dense}_{32}(\text{GlobalAvgPool}(\mathbf{H})) \quad \text{(compress to latent)}$$

**Decoder:**
$$\mathbf{H}' = \text{Reshape}(\text{Dense}_{50 \times 64}(\mathbf{z})) \in \mathbb{R}^{50 \times 64} \quad \text{(expand latent)}$$
$$\text{(symmetric Transformer decoder blocks)}$$
$$\hat{\mathbf{X}} = \text{Dense}_{34}(\mathbf{H}') \in \mathbb{R}^{50 \times 34} \quad \text{(reconstruct features)}$$

**Why Transformer > LSTM for long windows:** The LSTM processes flows strictly sequentially — information from flow 1 must pass through 49 recurrent steps to influence flow 50's encoding. At each step, even with gating, some information is compressed. The Transformer, by contrast, allows flow 1 to directly attend to flow 50 in a single operation. This **global receptive field** in $O(W^2)$ operations (vs. LSTM's $O(W \cdot d_h^2)$ sequential operations) makes the Transformer better at detecting **long-range temporal anomalies** — e.g., a change in flow behaviour in flows 45–50 that can only be understood in the context of a pattern established in flows 1–10.

### 1.4 — Regularization and Training Techniques

Careful regularisation is essential for our task: the training set is large (~150K sequences) but low in diversity — all sequences are benign CICIDS-2017 traffic. Without regularisation, the autoencoder risks two failure modes: (a) memorising the training sequences, producing a near-zero reconstruction error for *any* input (destroying anomaly sensitivity), or (b) converging to a trivial solution that ignores the input and outputs the mean sequence (destroying discriminativity).

#### A) Dropout

Dropout (Srivastava et al., 2014) randomly zeroes a fraction $p$ of activations during each forward pass of training:
$$\tilde{\mathbf{h}}_t = \mathbf{h}_t \odot \mathbf{m}_t, \quad \mathbf{m}_t \sim \text{Bernoulli}(1-p)^{d_h}$$

We apply **rate=0.2** (20% dropout) on all hidden-to-hidden connections in both LSTM layers and all Transformer feedforward sublayers. During inference, dropout is disabled and activations are scaled by $(1-p)$ to maintain expected magnitude.

**Why it prevents overfitting in our setting:** Dropout forces each neuron to learn useful features independently of any particular co-activating partner. For our benign training set — which has limited attack diversity by construction — this is critical: the network cannot rely on spurious correlations specific to CICIDS-2017 benign traffic, and must instead learn general structural properties of normal flow sequences. This improves generalisation to the attack distribution at test time.

**Specific concern:** Our 1.5M benign training flows yield ~150K sequences with $S=10$ stride. While this is a large sequence count, the underlying traffic comes from a single network capture over a few days — temporal autocorrelation means many sequences are near-duplicates. Dropout is the primary defence against overfitting to this correlated training signal.

#### B) Early Stopping

We hold out 20% of training sequences ($\sim$30K sequences) as a **validation set** that is never used for weight updates. Training stops when the validation reconstruction loss fails to improve for 10 consecutive epochs:

$$\text{Stop if: } \mathcal{L}_{\text{val}}(e) > \min_{e' < e-10} \mathcal{L}_{\text{val}}(e') \quad \forall e \in \{e_0+1, \ldots, e_0+10\}$$

**Restore best weights:** Upon stopping, we restore the model weights from the epoch $e^* = \arg\min_e \mathcal{L}_{\text{val}}(e)$ — the epoch with the lowest validation loss, not the final epoch. This is critical: the final epoch's weights may have begun overfitting (training loss still decreasing while validation loss has risen), and using them would inflate the anomaly scores on benign test sequences, degrading the threshold calibration.

Empirically, LSTM autoencoders on network traffic data typically converge between epochs 20–40 with early stopping, making the nominal 100-epoch maximum a safety bound rather than an expected runtime.

#### C) Weight Initialization

**LSTM weights** use **Xavier/Glorot uniform initialization** (Glorot & Bengio, 2010):
$$\mathbf{W} \sim \mathcal{U}\!\left(-\sqrt{\frac{6}{n_{\text{in}} + n_{\text{out}}}},\ +\sqrt{\frac{6}{n_{\text{in}} + n_{\text{out}}}}\right)$$

where $n_{\text{in}}$ and $n_{\text{out}}$ are the number of input and output units of the layer. Xavier initialization ensures that the variance of activations and gradients is approximately preserved through each layer, preventing the gradient signal from vanishing or exploding during the initial epochs of training.

**Forget gate bias initialization: $\mathbf{b}_f = \mathbf{1}$ (not $\mathbf{0}$).** This is a non-trivial implementation detail with significant practical impact. With $b_f = 0$, the forget gate starts at $\sigma(0) = 0.5$: the cell state is halved at every step, creating effective vanishing gradients in the early training phase before the gates are properly calibrated. With $b_f = 1$, the forget gate starts at $\sigma(1) \approx 0.73$: the cell state is preserved at 73% per step, giving gradients a clearer path through the cell state in early training. This practice was recommended by Jozefowicz et al. (2015) and is standard in high-quality LSTM implementations.

#### D) Gradient Clipping

We clip gradients by global norm with `clip_value = 1.0`:
$$\mathbf{g} \leftarrow \mathbf{g} \cdot \min\!\left(1, \frac{1.0}{\|\mathbf{g}\|_2}\right)$$

If the gradient norm exceeds 1.0, all gradients are rescaled proportionally so that the norm becomes exactly 1.0. Below the threshold, gradients are unchanged.

**Why needed for LSTMs:** Despite the cell state's gradient highway, exploding gradients can still occur in LSTMs when the input gate and output gate are simultaneously large, causing multiplicative amplification. For sequences of length $W=50$ with 34-dimensional inputs, sporadic gradient explosions can destabilise training — especially in early epochs before the gates are calibrated. Gradient clipping converts these explosive updates into safe, bounded weight changes.

#### E) Learning Rate Schedule

We use **ReduceLROnPlateau** with the following parameters:

| Parameter | Value | Rationale |
|-----------|-------|----------|
| Initial LR | $10^{-3}$ | Standard Adam starting point for sequence models |
| Factor | 0.5 | Halve LR on plateau |
| Patience | 5 epochs | Reduce after 5 epochs without val_loss improvement |
| Minimum LR | $10^{-6}$ | Floor to prevent LR from becoming negligibly small |

This schedule is complementary to early stopping: early stopping terminates training when fine-tuning is no longer improving generalisation, while ReduceLROnPlateau allows the optimiser to escape shallow local minima (by reducing LR to make finer gradient steps) before early stopping triggers. Typical behaviour: LR reduces from $10^{-3}$ to $5 \times 10^{-4}$ around epoch 15–20 as training loss plateaus, enabling a final phase of fine-tuning before early stopping at epoch 25–35.

### 1.5 — Sequence Construction Strategy

#### Sliding Window Formulation

Given a temporally ordered array of flow feature vectors $\mathbf{X} = [\mathbf{x}_1, \mathbf{x}_2, \ldots, \mathbf{x}_N]$ where $\mathbf{x}_i \in \mathbb{R}^F$, we construct overlapping windows of fixed length $W$:

$$\mathbf{X}^{(t)} = [\mathbf{x}_t,\ \mathbf{x}_{t+1},\ \ldots,\ \mathbf{x}_{t+W-1}] \in \mathbb{R}^{W \times F}$$

The $k$-th window starts at index $t_k = k \cdot S$ where $S$ is the stride, giving a total of $\lfloor (N - W) / S \rfloor + 1$ windows.

#### Parameter Choices

**Window size $W = 50$:** At typical enterprise network rates of ~10 flows per minute, a 50-flow window captures approximately 5 minutes of traffic. This is chosen to cover:
- Common C2 beacon intervals (typically 1–10 minutes)
- Short-burst DoS sub-patterns within a larger attack
- Multi-stage reconnaissance sequences (scan → probe → exploit)

A larger window (e.g., $W=200$) would capture more context but yield fewer training sequences and increase model complexity; $W=50$ is the sweet spot for our dataset size and target attack classes.

**Training stride $S_{\text{train}} = 10$:** Overlapping windows with stride 10 provide a **data augmentation** effect. From $N_{\text{benign}} \approx 1.52\text{M}$ benign training flows:
$$\text{Training sequences} = \left\lfloor \frac{1{,}518{,}344 - 50}{10} \right\rfloor + 1 \approx 151{,}830 \text{ sequences}$$

This substantially increases the diversity of training examples: each offset-10 window presents a slightly different slice of the temporal structure, preventing the model from memorising specific window positions.

**Test stride $S_{\text{test}} = 1$:** Dense evaluation with stride 1 ensures that every possible 50-flow subsequence in the test set receives a reconstruction error score. This maximises detection coverage: an attack that spans flows 25–75 will be captured by windows starting at any position $t \in \{25, 26, \ldots, 75\}$. With 49 overlapping windows covering the attack, even if early/late windows score below threshold, central windows (where the attack dominates the window) should exceed it.

#### Temporal Ordering

**Flow ordering** is critical for the validity of the sequence approach. Within the CICIDS-2017 dataset, flows are already ordered chronologically (Phase 1 preprocessing preserved the original dataset order). The Phase 1 preprocessing pipeline did not shuffle flows — it applied feature engineering and scaling in-place, preserving the temporal ordering that the sequence model depends on.

In a deployment setting, flows should be sorted by timestamp within each source-destination IP pair before windowing, preserving temporal causality (flows should be ordered as they occurred). However, for this dataset, the global temporal ordering provides sufficient structure for the sequence model.

#### Zero-Padding for Short Sequences

If a sequence has fewer than $W$ flows (e.g., at the end of the dataset), we apply **zero-padding** and a **binary mask**:
$$\mathbf{x}_t = \mathbf{0} \quad \forall t > N \quad \text{(padding)}$$
$$m_t = \mathbb{1}[t \leq N] \quad \text{(mask: 1=real, 0=padded)}$$

In practice, with $N_{\text{test}} = 716{,}092 \gg W = 50$, padding affects at most one window per dataset segment and is negligible for evaluation. We exclude padded windows from threshold calibration.

---
## Section 2: Architecture Specification

### 2.1 — LSTM Autoencoder Architecture

| Layer | Type | Units / Size | Activation | Notes |
|-------|------|-------------|------------|-------|
| Input | Input | (50, 34) | — | $W=50$ steps, $F=34$ features |
| Encoder LSTM 1 | LSTM | 128 units | tanh | `return_sequences=True` — passes full sequence to next layer |
| Encoder Dropout | Dropout | rate=0.2 | — | Applied to LSTM output activations |
| Encoder LSTM 2 | LSTM | 64 units | tanh | `return_sequences=False` — outputs final hidden state only |
| Latent | Dense | 32 units | ReLU | Bottleneck: $\mathbf{z} \in \mathbb{R}^{32}$; compresses $34 \times 50 = 1700 \to 32$ |
| Repeat | RepeatVector | 50 | — | Repeats $\mathbf{z}$ into shape $(50, 32)$ for decoder input |
| Decoder LSTM 1 | LSTM | 64 units | tanh | `return_sequences=True` — symmetric with encoder LSTM 2 |
| Decoder Dropout | Dropout | rate=0.2 | — | |
| Decoder LSTM 2 | LSTM | 128 units | tanh | `return_sequences=True` — symmetric with encoder LSTM 1 |
| Output | TimeDistributed(Dense) | 34 units | linear | Reconstructs each of the 50 time steps independently |

**Parameter count:**
- Encoder LSTM 1: $4 \times (128 \times (34 + 128) + 128) = 4 \times (128 \times 162 + 128) = 83{,}456$
- Encoder LSTM 2: $4 \times (64 \times (128 + 64) + 64) = 4 \times (64 \times 192 + 64) = 49{,}408$
- Latent Dense: $64 \times 32 + 32 = 2{,}080$
- Decoder LSTM 1: $4 \times (64 \times (32 + 64) + 64) = 4 \times (64 \times 96 + 64) = 24{,}832$
- Decoder LSTM 2: $4 \times (128 \times (64 + 128) + 128) = 4 \times (128 \times 192 + 128) = 98{,}816$
- Output Dense (TD): $(128 \times 34 + 34) = 4{,}386$
- **Total: ~263,000 trainable parameters**

**Training configuration:**
- Loss: Mean Squared Error (MSE)
- Optimizer: Adam($\text{lr}=10^{-3}$, $\beta_1=0.9$, $\beta_2=0.999$), gradient clip norm = 1.0
- Anomaly score: $s(\mathbf{X}) = \frac{1}{50}\sum_{t=1}^{50}\|\mathbf{x}_t - \hat{\mathbf{x}}_t\|_2^2$

---

### 2.2 — Transformer Autoencoder Architecture

| Layer | Type | Size | Notes |
|-------|------|------|-------|
| Input | Input | (50, 34) | $W=50$ steps, $F=34$ features |
| Positional Encoding | Sinusoidal PE | (50, 34) | Added (not concatenated) to input |
| Input Projection | Dense | $d_{\text{model}}=64$ | Project from 34 → 64 model dims |
| Enc TransformerBlock 1 | MultiHeadAttention + FFN + LayerNorm | heads=4, $d_k=16$ | Pre-LN variant; FFN hidden=128 |
| Enc TransformerBlock 2 | MultiHeadAttention + FFN + LayerNorm | heads=4, $d_k=16$ | Same structure |
| Encoder Dropout | Dropout | rate=0.2 | |
| Global Avg Pooling | GlobalAveragePooling1D | — | Compresses $(50, 64) \to (64,)$ |
| Latent | Dense | 32 units | Bottleneck $\mathbf{z} \in \mathbb{R}^{32}$ |
| Expansion | Dense | $50 \times 64 = 3200$ | Expand latent back |
| Reshape | Reshape | (50, 64) | Restore sequence shape |
| Dec TransformerBlock 1 | MultiHeadAttention + FFN + LayerNorm | heads=4, $d_k=16$ | Symmetric |
| Dec TransformerBlock 2 | MultiHeadAttention + FFN + LayerNorm | heads=4, $d_k=16$ | Symmetric |
| Output Projection | Dense | 34 units, linear | Reconstruct original features |

**Parameter count (approximate):**
- Input projection: $34 \times 64 + 64 = 2{,}240$
- Each TransformerBlock: MHA ($3 \times 64 \times 64 + 64 \times 64$) + FFN ($64 \times 128 + 128 + 128 \times 64 + 64$) $\approx 28{,}672$
- 4 blocks: $\approx 114{,}688$
- Latent + expansion layers: $64 \times 32 + 3200 \times 64 \approx 206{,}880$
- Output: $64 \times 34 + 34 = 2{,}210$
- **Total: ~326,000 trainable parameters**

**Training configuration:**
- Loss: Mean Squared Error (MSE)
- Optimizer: Adam($\text{lr}=10^{-3}$)
- Anomaly score: $s(\mathbf{X}) = \frac{1}{50}\sum_{t=1}^{50}\|\mathbf{x}_t - \hat{\mathbf{x}}_t\|_2^2$

---
## Section 3: Data Preparation Code

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
import tensorflow as tf

print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version: {np.__version__}")

# ── Constants ──────────────────────────────────────────────────────────────────
# W=50: at ~10 flows/min typical enterprise rate, captures ~5 min of traffic.
# This window covers common C2 beacon intervals and multi-stage attack sequences.
WINDOW_SIZE   = 50

# Stride=10 for training: overlapping windows provide data augmentation.
# ~1.5M flows / stride 10 ≈ 150K training sequences — sufficient for deep learning.
STRIDE_TRAIN  = 10

# Stride=1 for test: dense evaluation ensures no attack subsequence is missed.
# Every 50-flow sub-sequence in the test set receives a reconstruction error.
STRIDE_TEST   = 1

# Bottleneck dimension: 1700-dim sequence (50×34) → 32-dim latent.
# Compression ratio ~53× forces the network to learn a compact benign manifold.
LATENT_DIM    = 32

# Batch size 256: large enough for stable gradient estimates on MSE loss.
# Smaller batches (e.g., 32) would increase noise; larger (512+) may miss fine structure.
BATCH_SIZE    = 256

# 100 epochs is the upper bound; early stopping typically triggers at 20–40.
EPOCHS        = 100

RANDOM_STATE  = 42

# 20% of benign training sequences held out for early stopping and threshold calibration.
VAL_SPLIT     = 0.2

print("\nHyperparameter summary:")
print(f"  Window size      W = {WINDOW_SIZE}")
print(f"  Training stride  S = {STRIDE_TRAIN}")
print(f"  Test stride      S = {STRIDE_TEST}")
print(f"  Latent dim       d = {LATENT_DIM}")
print(f"  Batch size         = {BATCH_SIZE}")
print(f"  Max epochs         = {EPOCHS}")

In [ ]:
# ── Load Phase 1 preprocessed data ────────────────────────────────────────────
# Phase 1 pipeline: 5-stage preprocessing → RobustScaler → 34 features
# X_train: benign-only flows used for unsupervised training
# X_test:  benign + attack flows used for evaluation
# y_test:  binary labels (0=benign, 1=attack) aligned with X_test rows

PROJECT_ROOT = Path.cwd().parent
PREPROC_DIR  = PROJECT_ROOT / 'outputs' / 'preprocessing'
SEQ_DIR      = PROJECT_ROOT / 'outputs' / 'sequences'
SEQ_DIR.mkdir(parents=True, exist_ok=True)

X_train = np.load(PREPROC_DIR / 'X_train.npy')
X_test  = np.load(PREPROC_DIR / 'X_test.npy')
y_test  = np.load(PREPROC_DIR / 'y_test.npy')

with open(PREPROC_DIR / 'feature_names.txt') as f:
    feature_names = [line.strip() for line in f]

N_FEATURES = len(feature_names)

print("Phase 1 data loaded:")
print(f"  X_train : {X_train.shape}  (benign flows only)")
print(f"  X_test  : {X_test.shape}   (benign + attack flows)")
print(f"  y_test  : {y_test.shape}   distribution → {np.bincount(y_test.astype(int))} (0=benign, 1=attack)")
print(f"  Features: {N_FEATURES}     {feature_names[:5]} ...")
print(f"\n  Attack prevalence in test set: {y_test.mean()*100:.1f}%")

In [ ]:
# ── Sequence construction ──────────────────────────────────────────────────────

def make_sequences(X: np.ndarray, window_size: int, stride: int) -> np.ndarray:
    """
    Convert a flat flow array to overlapping fixed-length windows.

    Parameters
    ----------
    X           : (N, F) array of N flows, each with F features
    window_size : W — number of consecutive flows per sequence
    stride      : S — step between consecutive window start positions

    Returns
    -------
    sequences : (num_windows, W, F) float32 array

    Notes
    -----
    num_windows = floor((N - W) / S) + 1

    stride=10 for training increases sequence diversity via overlap.
    stride=1  for testing provides dense coverage — every 50-flow
    subsequence receives a reconstruction error score.

    Using stride_tricks for memory efficiency on large arrays.
    """
    N, F = X.shape
    num_windows = (N - window_size) // stride + 1

    # stride_tricks avoids copying data: each window is a view into X
    # shape: (num_windows, window_size, F)
    # strides: (stride * X.strides[0], X.strides[0], X.strides[1])
    shape   = (num_windows, window_size, F)
    strides = (stride * X.strides[0], X.strides[0], X.strides[1])
    windows = np.lib.stride_tricks.as_strided(X, shape=shape, strides=strides)

    # Copy to contiguous float32 array (required for TF dataset)
    return np.array(windows, dtype=np.float32)


def make_seq_labels(y: np.ndarray, window_size: int, stride: int) -> np.ndarray:
    """
    Derive sequence-level binary labels from flow-level labels.

    Labelling policy: a sequence X^(t) = [x_t, ..., x_{t+W-1}] is labelled
    as ANOMALOUS (1) if ANY of its constituent flows is an attack flow.

    Rationale: a sequence containing even a single attack flow is suspicious —
    the reconstruction error will be elevated by that flow's contribution.
    This is a conservative labelling that maximises recall at the sequence level.

    Parameters
    ----------
    y           : (N,) binary flow-level labels
    window_size : W
    stride      : S

    Returns
    -------
    labels : (num_windows,) int32 array, values in {0, 1}
    """
    N = len(y)
    num_windows = (N - window_size) // stride + 1
    labels = np.zeros(num_windows, dtype=np.int32)

    for k in range(num_windows):
        start = k * stride
        window_y = y[start : start + window_size]
        labels[k] = 1 if np.any(window_y == 1) else 0

    return labels


# Build sequences
print("Building sequences...")

X_train_seq = make_sequences(X_train, WINDOW_SIZE, STRIDE_TRAIN)
print(f"  Training sequences : {X_train_seq.shape}  (expected ~151,830)")
print(f"  Memory             : {X_train_seq.nbytes / 1e9:.2f} GB")

X_test_seq = make_sequences(X_test, WINDOW_SIZE, STRIDE_TEST)
print(f"  Test sequences     : {X_test_seq.shape}")
print(f"  Memory             : {X_test_seq.nbytes / 1e9:.2f} GB")

In [ ]:
# ── Test labels and train/val split ───────────────────────────────────────────
print("Building test sequence labels...")
y_test_seq = make_seq_labels(y_test, WINDOW_SIZE, STRIDE_TEST)

n_seq_benign = (y_test_seq == 0).sum()
n_seq_attack = (y_test_seq == 1).sum()
print(f"  Test sequence labels (ANY policy): {np.bincount(y_test_seq)}")
print(f"  Attack seq prevalence: {n_seq_attack / len(y_test_seq) * 100:.1f}%")

# ── IMPORTANT: Test set shuffle note ─────────────────────────────────────────
# Phase 1 used sklearn's train_test_split (shuffle=True by default), so X_test
# flows are in RANDOM order — not temporal order. With ~47% attack prevalence,
# virtually every 50-flow window contains >=1 attack flow
# (P(all benign in window) ~ 0.53^50 ~ 1e-14). So ANY-policy labels are ~all 1.
#
# Evaluation strategy for reconstruction-based anomaly detection:
#   1. Per-FLOW score: for each flow i, average the reconstruction error
#      of all windows containing flow i (windows t: t <= i <= t+W-1).
#      This maps the window-level score back to flow-level, enabling
#      ROC/AUC computed against the original y_test (N=716K flows).
#
#   2. Score distribution: compare reconstruction errors for windows
#      that are 100% benign vs windows with >=50% attack flows.
#
# We save a continuous "attack fraction" per window for richer evaluation
# in Stage 5 (model training & evaluation notebook).

def make_seq_attack_fraction(y, window_size, stride):
    """
    Compute the fraction of attack flows in each window.
    Returns float32 array in [0, 1]. More informative than binary label
    when test flows are randomly shuffled.
    """
    N = len(y)
    num_windows = (N - window_size) // stride + 1
    fracs = np.zeros(num_windows, dtype=np.float32)
    for k in range(num_windows):
        start = k * stride
        fracs[k] = float(y[start : start + window_size].mean())
    return fracs

print("\nComputing attack fraction per test window...")
y_test_seq_frac = make_seq_attack_fraction(y_test, WINDOW_SIZE, STRIDE_TEST)

pure_benign_mask  = y_test_seq_frac == 0.0
pure_attack_mask  = y_test_seq_frac == 1.0
mixed_mask        = (y_test_seq_frac > 0) & (y_test_seq_frac < 1)
majority_att_mask = y_test_seq_frac >= 0.5

print(f"\n  Window composition (total {len(y_test_seq_frac):,}):")
print(f"    Pure benign  (0% attack)  : {pure_benign_mask.sum():>8,}  ({pure_benign_mask.mean()*100:.1f}%)")
print(f"    Pure attack  (100% attack): {pure_attack_mask.sum():>8,}  ({pure_attack_mask.mean()*100:.1f}%)")
print(f"    Mixed windows             : {mixed_mask.sum():>8,}  ({mixed_mask.mean()*100:.1f}%)")
print(f"    >=50%% attack              : {majority_att_mask.sum():>8,}  ({majority_att_mask.mean()*100:.1f}%)")
print("\n  => Because test flows are randomly shuffled from Phase 1,")
print("     pure-benign windows are rare. Primary evaluation will use")
print("     per-FLOW AUC: aggregate window scores back to flow level.")

# Train/validation split (all training sequences are benign)
print("\nSplitting training sequences into train / validation...")
X_tr, X_val = train_test_split(
    X_train_seq, test_size=VAL_SPLIT, random_state=RANDOM_STATE, shuffle=True
)
print(f"  Train : {X_tr.shape}")
print(f"  Val   : {X_val.shape}")


In [ ]:
# ── Save sequences to disk ────────────────────────────────────────────────────
print("Saving sequences to outputs/sequences/...")

np.save(SEQ_DIR / 'X_train_seq.npy', X_tr)
np.save(SEQ_DIR / 'X_val_seq.npy',   X_val)
np.save(SEQ_DIR / 'X_test_seq.npy',  X_test_seq)
np.save(SEQ_DIR / 'y_test_seq.npy',  y_test_seq)
np.save(SEQ_DIR / 'y_test_seq_frac.npy', y_test_seq_frac)

# Save metadata for downstream notebooks
import pickle
meta = {
    'window_size':   WINDOW_SIZE,
    'stride_train':  STRIDE_TRAIN,
    'stride_test':   STRIDE_TEST,
    'latent_dim':    LATENT_DIM,
    'batch_size':    BATCH_SIZE,
    'epochs':        EPOCHS,
    'val_split':     VAL_SPLIT,
    'n_features':    N_FEATURES,
    'n_train_seq':   len(X_tr),
    'n_val_seq':     len(X_val),
    'n_test_seq':    len(X_test_seq),
    'n_attack_seq':  int(n_seq_attack),
    'n_benign_seq':  int(n_seq_benign),
    'note_shuffle':  (
        "Test flows are randomly shuffled from Phase 1 train_test_split. "
        "Evaluation uses per-flow AUC (aggregate window scores to flow level) "
        "rather than per-sequence binary labels."
    ),
}
with open(SEQ_DIR / 'sequence_metadata.pkl', 'wb') as f:
    pickle.dump(meta, f)

print("\nSaved:")
for f in sorted(SEQ_DIR.iterdir()):
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name:<35} {size_mb:6.1f} MB")

print("\n✓ Data preparation complete. Ready for Stage 5 model building.")
print("\n  Key note: evaluation in Stage 5 will use per-flow AUC by")
print("  mapping window reconstruction errors back to individual flows.")


---
## Section 4: Architecture Diagram

In [ ]:
# Set non-interactive backend if running without display
import os
if os.environ.get('DISPLAY') is None and os.name != 'nt':
    import matplotlib
    matplotlib.use('Agg')

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import matplotlib.patheffects as pe
import warnings
warnings.filterwarnings('ignore')

# ── Visual style — match Phase 1 architecture diagram ─────────────────────────
matplotlib.rcParams.update({
    'font.family':      'DejaVu Sans',
    'font.size':        9,
    'axes.titlesize':   11,
    'figure.facecolor': '#FAFAFA',
    'savefig.dpi':      150,
    'savefig.bbox':     'tight',
})

# Colour palette
C_INPUT    = '#AED6F1'   # light blue  — input layers
C_ENCODER  = '#A9DFBF'   # light green — encoder layers
C_LATENT   = '#F9E79F'   # yellow      — bottleneck
C_DECODER  = '#F1948A'   # salmon      — decoder layers
C_OUTPUT   = '#D2B4DE'   # lavender    — output / score
C_PHASE1   = '#D5D8DC'   # light grey  — Phase 1 GMM (greyed out)
C_PHASE3   = '#D5D8DC'   # light grey  — Phase 3 Hybrid (future)
C_EDGE     = '#2C3E50'   # dark navy   — arrows
C_DASHED   = '#7F8C8D'   # grey        — dashed future components

fig, ax = plt.subplots(figsize=(18, 13))
ax.set_xlim(0, 18)
ax.set_ylim(0, 13)
ax.axis('off')
fig.patch.set_facecolor('#FAFAFA')

# ─────────────────────────────────────────────────────────────────────────────
# Helper functions
# ─────────────────────────────────────────────────────────────────────────────

def box(ax, x, y, w, h, label, sublabel='', color='#FFFFFF',
        fontsize=8.5, bold=False, alpha=1.0, linestyle='solid'):
    rect = FancyBboxPatch(
        (x - w/2, y - h/2), w, h,
        boxstyle='round,pad=0.05',
        facecolor=color, edgecolor=C_EDGE if linestyle == 'solid' else C_DASHED,
        linewidth=1.4, alpha=alpha,
        linestyle=linestyle
    )
    ax.add_patch(rect)
    weight = 'bold' if bold else 'normal'
    ax.text(x, y + (0.1 if sublabel else 0), label,
            ha='center', va='center', fontsize=fontsize,
            fontweight=weight, color='#2C3E50', alpha=alpha)
    if sublabel:
        ax.text(x, y - 0.22, sublabel,
                ha='center', va='center', fontsize=7,
                color='#566573', alpha=alpha)


def arrow(ax, x1, y1, x2, y2, color=C_EDGE, lw=1.5,
          arrowstyle='->', linestyle='solid', alpha=1.0):
    ax.annotate("",
        xy=(x2, y2), xytext=(x1, y1),
        arrowprops=dict(
            arrowstyle=arrowstyle,
            color=color,
            lw=lw,
            linestyle=linestyle,
            alpha=alpha,
            connectionstyle='arc3,rad=0'
        )
    )


def section_label(ax, x, y, text, color):
    ax.text(x, y, text, ha='center', va='center',
            fontsize=12, fontweight='bold', color=color,
            bbox=dict(boxstyle='round,pad=0.3', facecolor=color,
                      edgecolor=color, alpha=0.15))


# ─────────────────────────────────────────────────────────────────────────────
# Title
# ─────────────────────────────────────────────────────────────────────────────
ax.text(9, 12.5,
        'Phase 2 — Deep Learning Anomaly Detection Architecture',
        ha='center', va='center', fontsize=14, fontweight='bold', color='#2C3E50')
ax.text(9, 12.0,
        'LSTM Autoencoder (left) vs. Transformer Autoencoder (right)  |  '
        'Input: sliding windows of W=50 flows, F=34 features',
        ha='center', va='center', fontsize=9, color='#566573')

# ─────────────────────────────────────────────────────────────────────────────
# LEFT SIDE — LSTM Autoencoder
# ─────────────────────────────────────────────────────────────────────────────
LX = 4.0   # centre x of left column

section_label(ax, LX, 11.4, 'LSTM Autoencoder (Model B₁)', '#1A5276')

layer_specs_lstm = [
    # (y, label, sublabel, color)
    (10.55, 'Input Sequence',       '(50, 34)  W=50 flows × 34 features', C_INPUT),
    (9.55,  'Encoder LSTM 1',       '128 units | return_seq=True | Dropout 0.2', C_ENCODER),
    (8.55,  'Encoder LSTM 2',       '64 units | return_seq=False', C_ENCODER),
    (7.55,  'Latent z',             'Dense(32, ReLU) | 1700 → 32 dims', C_LATENT),
    (6.55,  'RepeatVector(50)',      'Expand z back to sequence length', C_LATENT),
    (5.55,  'Decoder LSTM 1',       '64 units | return_seq=True | Dropout 0.2', C_DECODER),
    (4.55,  'Decoder LSTM 2',       '128 units | return_seq=True', C_DECODER),
    (3.55,  'TimeDistributed Dense','Dense(34, linear) → X̂ ∈ ℝ^{50×34}', C_OUTPUT),
    (2.55,  'Anomaly Score',        's(X) = (1/50) Σ‖xᵢ − x̂ᵢ‖²', C_OUTPUT),
    (1.55,  'Threshold τ',          'μ_benign + k·σ_benign | k tuned on val', C_OUTPUT),
]

for (y, lbl, sub, col) in layer_specs_lstm:
    box(ax, LX, y, 5.5, 0.70, lbl, sub, color=col)

# Arrows between LSTM layers
for i in range(len(layer_specs_lstm) - 1):
    y_top = layer_specs_lstm[i][0] - 0.35
    y_bot = layer_specs_lstm[i+1][0] + 0.35
    arrow(ax, LX, y_top, LX, y_bot)

# Encoder/Decoder bracket annotations
ax.annotate('', xy=(1.0, 8.2), xytext=(1.0, 10.9),
            arrowprops=dict(arrowstyle='<->', color=C_ENCODER, lw=1.5))
ax.text(0.55, 9.55, 'ENCODER', va='center', ha='center', fontsize=8,
        color=C_ENCODER, fontweight='bold', rotation=90)

ax.annotate('', xy=(1.0, 3.2), xytext=(1.0, 6.9),
            arrowprops=dict(arrowstyle='<->', color=C_DECODER, lw=1.5))
ax.text(0.55, 5.05, 'DECODER', va='center', ha='center', fontsize=8,
        color=C_DECODER, fontweight='bold', rotation=90)

# ─────────────────────────────────────────────────────────────────────────────
# RIGHT SIDE — Transformer Autoencoder
# ─────────────────────────────────────────────────────────────────────────────
RX = 14.0   # centre x of right column

section_label(ax, RX, 11.4, 'Transformer Autoencoder (Model B₂)', '#6E2F8E')

layer_specs_tfm = [
    (10.55, 'Input Sequence',         '(50, 34)  W=50 flows × 34 features', C_INPUT),
    (9.55,  'Positional Encoding',    'Sinusoidal PE added to input embeddings', C_INPUT),
    (8.65,  'Input Projection',       'Dense(64) → d_model=64', C_ENCODER),
    (7.80,  'Enc TransformerBlock 1', 'MHA(heads=4, dₖ=16) + FFN(128) + LayerNorm', C_ENCODER),
    (6.95,  'Enc TransformerBlock 2', 'MHA(heads=4, dₖ=16) + FFN(128) + LayerNorm | Dropout 0.2', C_ENCODER),
    (6.10,  'Global Avg Pool → Latent z', 'GlobalAvgPool1D → Dense(32, ReLU) | 1700→32', C_LATENT),
    (5.25,  'Latent Expansion',       'Dense(3200) → Reshape(50, 64)', C_LATENT),
    (4.40,  'Dec TransformerBlock 1', 'MHA(heads=4, dₖ=16) + FFN(128) + LayerNorm', C_DECODER),
    (3.55,  'Dec TransformerBlock 2', 'MHA(heads=4, dₖ=16) + FFN(128) + LayerNorm', C_DECODER),
    (2.70,  'Output Projection',      'Dense(34, linear) → X̂ ∈ ℝ^{50×34}', C_OUTPUT),
    (1.85,  'Anomaly Score + τ',      's(X) = (1/50) Σ‖xᵢ − x̂ᵢ‖²  |  τ = μ + k·σ', C_OUTPUT),
]

for (y, lbl, sub, col) in layer_specs_tfm:
    box(ax, RX, y, 5.5, 0.63, lbl, sub, color=col)

for i in range(len(layer_specs_tfm) - 1):
    y_top = layer_specs_tfm[i][0] - 0.315
    y_bot = layer_specs_tfm[i+1][0] + 0.315
    arrow(ax, RX, y_top, RX, y_bot)

# Attention symbol annotations on Transformer blocks
for y in [7.80, 6.95, 4.40, 3.55]:
    ax.text(RX + 2.9, y, '⚡', fontsize=11, va='center', ha='center', alpha=0.6)

# Encoder/Decoder bracket annotations (Transformer)
ax.annotate('', xy=(17.1, 5.8), xytext=(17.1, 10.9),
            arrowprops=dict(arrowstyle='<->', color=C_ENCODER, lw=1.5))
ax.text(17.55, 8.35, 'ENCODER', va='center', ha='center', fontsize=8,
        color=C_ENCODER, fontweight='bold', rotation=90)

ax.annotate('', xy=(17.1, 1.5), xytext=(17.1, 5.1),
            arrowprops=dict(arrowstyle='<->', color=C_DECODER, lw=1.5))
ax.text(17.55, 3.3, 'DECODER', va='center', ha='center', fontsize=8,
        color=C_DECODER, fontweight='bold', rotation=90)

# ─────────────────────────────────────────────────────────────────────────────
# CENTRE — Phase 1 GMM + Phase 3 Hybrid (greyed out)
# ─────────────────────────────────────────────────────────────────────────────

# Phase 1 GMM box (grey, dashed)
box(ax, 9.0, 7.55, 2.8, 0.70,
    'Phase 1 — GMM', 'K=12 | F1=90.97% | per-flow score',
    color=C_PHASE1, alpha=0.7, linestyle='dashed')
ax.text(9.0, 8.3, 'Baseline Model A', ha='center', va='center',
        fontsize=7.5, color=C_DASHED, style='italic')

# Arrows from both models and Phase 1 to Phase 3
# Left model score → centre
arrow(ax, LX + 2.75, 1.55, 7.55, 3.15, color=C_DASHED, lw=1.2,
      linestyle='dashed', alpha=0.6)
# Right model score → centre
arrow(ax, RX - 2.75, 1.85, 10.45, 3.15, color=C_DASHED, lw=1.2,
      linestyle='dashed', alpha=0.6)
# Phase 1 GMM → centre
arrow(ax, 9.0, 7.2, 9.0, 4.2, color=C_DASHED, lw=1.2,
      linestyle='dashed', alpha=0.6)

# Phase 3 Hybrid box (grey, dashed)
box(ax, 9.0, 3.55, 2.8, 0.85,
    '⟶ Phase 3: Hybrid Ensemble ⟵', 'GMM score + DL score → fusion layer',
    color=C_PHASE3, alpha=0.6, linestyle='dashed')
ax.text(9.0, 2.95, '(Future work — Phase 3)', ha='center', va='center',
        fontsize=7, color=C_DASHED, style='italic')

# ─────────────────────────────────────────────────────────────────────────────
# Shared input annotation
# ─────────────────────────────────────────────────────────────────────────────
box(ax, 9.0, 10.55, 2.6, 0.62,
    'Sliding Window Input', 'X^(t) ∈ ℝ^{50×34}  |  S=10 train, S=1 test',
    color='#D6EAF8', bold=False)

arrow(ax, 7.7, 10.55, LX + 2.75, 10.55, color='#2980B9', lw=1.3)
arrow(ax, 10.3, 10.55, RX - 2.75, 10.55, color='#8E44AD', lw=1.3)
arrow(ax, 9.0, 10.24, 9.0, 7.9, color=C_DASHED, lw=1.1,
      linestyle='dashed', alpha=0.55)

# ─────────────────────────────────────────────────────────────────────────────
# Legend
# ─────────────────────────────────────────────────────────────────────────────
legend_items = [
    (C_INPUT,   'Input / Preprocessing'),
    (C_ENCODER, 'Encoder layers'),
    (C_LATENT,  'Bottleneck / Latent'),
    (C_DECODER, 'Decoder layers'),
    (C_OUTPUT,  'Output / Anomaly scoring'),
    (C_PHASE1,  'Baseline / Future (greyed out)'),
]
handles = [
    mpatches.Patch(facecolor=c, edgecolor=C_EDGE, label=lbl, linewidth=0.8)
    for c, lbl in legend_items
]
ax.legend(handles=handles, loc='lower center', ncol=3,
          bbox_to_anchor=(0.5, -0.02),
          fontsize=8, framealpha=0.85, edgecolor='#BDC3C7')

# ─────────────────────────────────────────────────────────────────────────────
# Footer
# ─────────────────────────────────────────────────────────────────────────────
ax.text(9.0, 0.55,
        'Model B candidates — evaluated and compared in Section 3 (Stage 5)  '
        '|  Best Model B advances to Phase 3 hybrid fusion with Model A (GMM)',
        ha='center', va='center', fontsize=8.5, color='#566573',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#EBF5FB',
                  edgecolor='#AED6F1', linewidth=1))

# ─────────────────────────────────────────────────────────────────────────────
# Save
# ─────────────────────────────────────────────────────────────────────────────
OUT_DIR = PROJECT_ROOT / 'outputs'

fig.savefig(OUT_DIR / 'architecture_diagram_phase2.png', dpi=150, bbox_inches='tight',
            facecolor='#FAFAFA')
fig.savefig(OUT_DIR / 'architecture_diagram_phase2.pdf', bbox_inches='tight',
            facecolor='#FAFAFA')

plt.tight_layout()
plt.show()
print("\nSaved: outputs/architecture_diagram_phase2.{png,pdf}")

---
## Phase 2 Rationale

*The following 200-word rationale is written for direct inclusion in the Phase 2 LaTeX report introduction.*

---

Phase 1 established a strong baseline for network intrusion detection using a Gaussian Mixture Model (GMM) with $K=12$ components, achieving F1=90.97% and AUC=95.76% on the CICIDS-2017 benchmark. However, a systematic failure analysis revealed a critical architectural limitation: the GMM models each network flow as an independent draw from a stationary distribution, violating the fundamental temporal structure of network traffic. This IID assumption is most severely violated for **low-and-slow attacks**, which deliberately distribute their malicious signatures across many flows to evade per-flow detectors. Empirically, Phase 1 detected Bot attacks at only 45.2% and Infiltration attacks at 33.1% — the two attack classes most characterised by long-horizon temporal patterns.

Phase 2 directly addresses this gap by moving from **flow space** to **sequence space**. Both proposed models — the LSTM Autoencoder and Transformer Autoencoder — process sliding windows of $W=50$ consecutive flows as a single unit, learning the joint distribution of benign flow sequences. Attacks that appear individually normal are exposed by their collective failure to conform to the temporal regularities encoded during training. The LSTM Autoencoder brings sequential memory via gated cell states; the Transformer Autoencoder brings global temporal attention, capturing long-range dependencies that the LSTM's sequential bottleneck cannot. Together, they form two complementary hypotheses about the structure of temporal benign normality. The stronger of the two advances to Phase 3 as Model B for hybrid fusion with the Phase 1 GMM baseline.